In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive/')
dir_path = '/content/drive/MyDrive/02740_data/dimension_reduction' # Path to the folder

os.chdir(dir_path)

# Dimensionality Reduction for Bioimage Analysis

### What we are going to learn ?
- 1. We are going to write code for dimensionality reduction.
- 2. Learn how to use dimensionality reduction for Bio-image analysis

## Principal Component Analysis (PCA)


Principal Component Analysis (PCA) is a widely used dimensionality reduction technique that transforms high-dimensional data into a lower-dimensional form while retaining as much of the original variance as possible. PCA achieves this by identifying the principal components, which are the directions in the data that capture the maximum variance. These components are orthogonal to each other and represent the most significant features of the data. PCA is commonly used in data preprocessing, visualization, and noise reduction, and it helps in simplifying complex datasets by highlighting their most important aspects.

### PCA for Image Compression / Reconstruction

**Dataset**: DRIVE: Digital Retinal Images for Vessel Extraction; Used for segmentation of blood vessels in retinal images

### Normalization

Dataset normalization is a crucial preprocessing step before applying Principal Component Analysis (PCA). Since PCA is sensitive to the scale of the data, normalizing the dataset ensures that each feature contributes equally to the analysis. Without normalization, features with larger scales could disproportionately influence the principal components, leading to biased results. Typically, normalization involves standardizing the features by subtracting the mean and dividing by the standard deviation, resulting in a dataset with a mean of zero and a standard deviation of one. This process ensures that PCA captures the true structure of the data, allowing for meaningful dimensionality reduction.


In [ ]:
import cv2
import os
import numpy as np
from sklearn.preprocessing import StandardScaler
from torchvision import transforms
from PIL import Image

data_dir = './training/images/'

# Load images into a list or array
images = []
for img_name in os.listdir(data_dir):
    img_path = os.path.join(data_dir, img_name)
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # Loading as grayscale
    images.append(img)


augment = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
])

# Apply augmentation to each image
augmented_images = []
for img in images:
    img_pil = Image.fromarray(img)
    augmented_img = augment(img_pil)
    augmented_images.append(np.array(augmented_img))

augmented_images = np.array(augmented_images)
images = np.array(images)

combined_images = np.concatenate((augmented_images, images), axis=0)

num_images, height, width = combined_images.shape
flat_images = combined_images.reshape(num_images, height * width)

scaler = StandardScaler()
flat_images_normalized = scaler.fit_transform(flat_images)


### PCA using Sklearn


In [ ]:
from sklearn.decomposition import PCA

# Apply PCA to reduce the dimensionality of the scaled data
# Number of components to keep, e.g., 20 components
pca = PCA(n_components=40)

pca_images = pca.fit_transform(flat_images_normalized)


### Reconstructed Image


In [ ]:
reconstructed_images = pca.inverse_transform(pca_images)

# Reshape the 1D arrays back to 2D image format
reconstructed_images = reconstructed_images.reshape(num_images, height, width)

# Visualize the first reconstructed image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(reconstructed_images[0], cmap='gray')
axes[0].set_title('Reconstructed Image')

axes[1].imshow(flat_images_normalized[0].reshape(height, width), cmap='gray')
axes[1].set_title('Original Image')

plt.show()

In [ ]:
pca_2d = PCA(n_components=2)
pca_2d_embedding = pca_2d.fit_transform(flat_images_normalized)

plt.scatter(pca_2d_embedding[:, 0], pca_2d_embedding[:, 1], s=50, cmap='viridis')
plt.title('2D PCA Embedding of DRIVEdataset')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()

### Feature-based Dimensionality Reduction

In [ ]:
import pandas as pd
feat_mat = pd.read_csv('features_df_imputed.csv')
labels = pd.read_csv('labels.csv')
feat_mat.head()

In [ ]:
scaler = StandardScaler()
feat_mat_norm = scaler.fit_transform(feat_mat)
pca_2d = PCA(n_components=2)
feat_embed_pca = pca_2d.fit_transform(feat_mat_norm)
feat_embed_pca_df = pd.DataFrame(feat_embed_pca, columns=['PCA1', 'PCA2'])
feat_embed_pca_df['Label'] = labels
feat_embed_pca_df.head()

In [ ]:
import seaborn as sns

num_classes = len(np.unique(labels))
palette = sns.color_palette("hsv", num_classes)

plt.figure(figsize=(10, 8))

# PCA Scatter Plot
sns.scatterplot(
    x='PCA1', y='PCA2',
    hue='Label',
    palette=palette,
    data=feat_embed_pca_df,
    legend='full',
    alpha=0.7
)
plt.title('PCA Projection')
plt.xlabel('PCA1')
plt.ylabel('PCA2')

# Limitations of PCA
- PCA may miss non-linear data patterns
- Structure that is not orthogonal to previous PCs may not be will characterized

# Multi-Dimensional Scaling (MDS)
- Multi-Dimensional Scaling (MDS) is a dimensionality reduction technique that aims to preserve the pairwise distances between data points when projecting them into a lower-dimensional space.

In [ ]:
# Import necessary libraries and functions
from sklearn.manifold import MDS
from sklearn.metrics import euclidean_distances

# Compute the Euclidean distances between the samples
similarities = euclidean_distances(feat_mat_norm)

# Initialize and fit MDS with precomputed dissimilarities (Euclidean distances)
mds = MDS(n_components=2, max_iter=3000, eps=1e-9, random_state=0, dissimilarity='precomputed', n_jobs=1)
pos = mds.fit_transform(similarities)
print("POS :", pos)


In [ ]:
feat_embed_mds_df = pd.DataFrame(pos, columns=['MDS1', 'MDS2'])
feat_embed_mds_df['Label'] = labels

plt.figure(figsize=(10, 8))

# MDS Scatter Plot
sns.scatterplot(
    x='MDS1', y='MDS2',
    hue='Label',
    palette=palette,
    data=feat_embed_mds_df,
    legend='full',
    alpha=0.7
)
plt.title('MDS Projection')
plt.xlabel('MDS1')
plt.ylabel('MDS2')

# Isometric Feature Mapping(ISOMAP)
- This is a graph based algorthim constructing a matrix of "geodesic" distances
- Geodesic Distances as a measure of proximity between the multi-dimensional points on the manifold approximated by the K-NN graph


In [ ]:
# Import necessary libraries
from sklearn import manifold

isomap = manifold.Isomap(n_neighbors=4, n_components=2)
feat_embed_isomap = isomap.fit_transform(feat_mat_norm)
feat_embed_isomap_df = pd.DataFrame(feat_embed_isomap, columns=['ISOMAP1', 'ISOMAP2'])
feat_embed_isomap_df['Label'] = labels

plt.figure(figsize=(10, 8))

# MDS Scatter Plot
sns.scatterplot(
    x='ISOMAP1', y='ISOMAP2',
    hue='Label',
    palette=palette,
    data=feat_embed_isomap_df,
    legend='full',
    alpha=0.7
)
plt.title('ISOMAP Projection')
plt.xlabel('ISOMAP1')
plt.ylabel('ISOMAP2')


# t-Distributed Stochastic Neighbor Embedding(t-SNE)
- A nonlinear dimensionality reduction technique that visualizes high-dimensional data by embedding it in a lower-dimensional space while preserving local similarities.
- Unlike PCA, which is a linear method preserving global structures, t-SNE is a nonlinear technique focused on preserving local structures in the data.



In [ ]:
# Import necessary libraries
from sklearn.manifold import TSNE

# Apply t-SNE
tsne = TSNE(n_components=2, random_state=42)
feat_embed_tsne = tsne.fit_transform(feat_mat_norm)
feat_embed_tsne_df = pd.DataFrame(feat_embed_tsne, columns=['TSNE1', 'TSNE2'])
feat_embed_tsne_df['Label'] = labels

plt.figure(figsize=(10, 8))


sns.scatterplot(
    x='TSNE1', y='TSNE2',
    hue='Label',
    palette=palette,
    data=feat_embed_tsne_df,
    legend='full',
    alpha=0.7
)



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 15))

# PCA Scatter Plot
sns.scatterplot(
    x='PCA1', y='PCA2',
    hue='Label',
    palette=palette,
    data=feat_embed_pca_df,
    legend='full',
    alpha=0.7,
    ax=axes[0, 0]
)
axes[0, 0].set_title('PCA Projection')
axes[0, 0].set_xlabel('PCA1')
axes[0, 0].set_ylabel('PCA2')

# MDS Scatter Plot
sns.scatterplot(
    x='MDS1', y='MDS2',
    hue='Label',
    palette=palette,
    data=feat_embed_mds_df,
    legend='full',
    alpha=0.7,
    ax=axes[0, 1]
)
axes[0, 1].set_title('MDS Projection')
axes[0, 1].set_xlabel('MDS1')
axes[0, 1].set_ylabel('MDS2')


# ISOMAP Scatter Plot
sns.scatterplot(
    x='ISOMAP1', y='ISOMAP2',
    hue='Label',
    palette=palette,
    data=feat_embed_isomap_df,
    legend='full',
    alpha=0.7,
    ax=axes[1, 0]
)
axes[1, 0].set_title('ISOMAP Projection')
axes[1, 0].set_xlabel('ISOMAP1')
axes[1, 0].set_ylabel('ISOMAP2')

# t-SNE Scatter Plot
sns.scatterplot(
    x='TSNE1', y='TSNE2',
    hue='Label',
    palette=palette,
    data=feat_embed_tsne_df,
    legend='full',
    alpha=0.7,
    ax=axes[1, 1]
)
axes[1, 1].set_title('t-SNE Projection')
axes[1, 1].set_xlabel('TSNE1')
axes[1, 1].set_ylabel('TSNE2')

plt.tight_layout()
plt.show()

# AutoEncoders

Autoencoders are a type of artificial neural network used for unsupervised learning. They are designed to learn efficient codings of input data by compressing the input into a latent space representation and then reconstructing the output from this representation. The network consists of two main parts: an encoder, which compresses the input, and a decoder, which reconstructs the input from the compressed code. Autoencoders are widely used for dimensionality reduction, denoising, anomaly detection, and feature learning. By learning to reconstruct the input data, autoencoders can capture the most important features in the data, making them a powerful tool for representation learning.








In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive/')
dir_path = '/content/drive/MyDrive/02740_data/dimension_reduction' # Path to the folder

os.chdir(dir_path)

In [ ]:
import os
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import cv2


class DRIVEDataset(Dataset):
    def __init__(self, img_dir):
        self.img_dir = img_dir
        self.image_files = [f for f in os.listdir(img_dir) if f.endswith('.tif')]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.image_files[idx])

        # Load the image using OpenCV (or PIL)
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        image = image / 255.0

        image = cv2.resize(image, (256, 256))

        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)  # Add channel dimension

        return image


In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=2, padding=1),  # Increase channels (64, 128, etc.)
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1)  # Add more channels for richer representation
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 1, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()  # Output in the range [0, 1]
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


In [ ]:
img_dir = './training/images/'
batch_size = 1
num_epochs = 20
learning_rate = 0.001

train_dataset = DRIVEDataset(img_dir)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

model = Autoencoder().to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
criterion = nn.MSELoss()  # Mean squared error for reconstruction
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

def train_autoencoder(model, dataloader, num_epochs):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for data in dataloader:
            data = data.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

            # Forward pass: autoencoder reconstructs the input
            outputs = model(data)
            loss = criterion(outputs, data)

            # Backward pass: optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(dataloader):.4f}")

def visualize_reconstruction(model, dataloader):
    model.eval()
    with torch.no_grad():
        for data in dataloader:
            data = data.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
            outputs = model(data)

            # Display the first image in the batch (original and reconstructed)
            fig, axs = plt.subplots(1, 2, figsize=(10, 5))
            axs[0].imshow(data[0].cpu().numpy().squeeze(), cmap='gray')
            axs[0].set_title("Original Image")
            axs[1].imshow(outputs[0].cpu().numpy().squeeze(), cmap='gray')
            axs[1].set_title("Reconstructed Image")
            plt.show()
            break

In [ ]:
train_autoencoder(model, train_loader, num_epochs)


In [ ]:
visualize_reconstruction(model, train_loader)
